In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle
import os

df = pd.read_csv('../data/property_listings.csv')
print(f"Raw shape: {df.shape}")

Raw shape: (105, 8)


In [5]:
import re

# Price
df['price_clean'] = (
    df['Price']
    .str.replace('KSh', '', regex=False)
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(' ', '', regex=False)
    .str.strip()
)
df['price_clean'] = pd.to_numeric(df['price_clean'], errors='coerce')

# Floor size
df['floor_size_clean'] = (
    df['Floor Size']
    .str.replace('m²', '', regex=False)
    .str.replace('m2', '', regex=False)
    .str.strip()
)
df['floor_size_clean'] = pd.to_numeric(df['floor_size_clean'], errors='coerce')

# Dedup
df = df.drop_duplicates(subset=['Title', 'Price', 'Address', 'Floor Size'])

# Property type
df['property_type'] = df['Title'].str.extract(
    r'(Apartment|House|Villa|Townhouse|Studio|Penthouse)', flags=re.IGNORECASE
)
df['property_type'] = df['property_type'].fillna('Other').str.title()

# Price per sqm
df['price_per_sqm'] = df['price_clean'] / df['floor_size_clean']

# Bedrooms + bathrooms
df['bedrooms'] = pd.to_numeric(df['Bedrooms'], errors='coerce')
df['bathrooms'] = df['Bathrooms']

print(f"Clean shape: {df.shape}")

Clean shape: (66, 14)


In [6]:
# One-Hot encode Location
location_dummies = pd.get_dummies(df['Location'], prefix='loc')
df = pd.concat([df, location_dummies], axis=1)
loc_cols = [c for c in df.columns if c.startswith('loc_')]

# Label encode property type
le_type = LabelEncoder()
df['type_encoded'] = le_type.fit_transform(df['property_type'])

# Define features AFTER all encoding is done
feature_cols = ['bedrooms', 'bathrooms', 'floor_size_clean', 'type_encoded'] + loc_cols

print("Features:", feature_cols)
print("\nProperty type encoding:")
for i, label in enumerate(le_type.classes_):
    print(f"  {i} → {label}")

Features: ['bedrooms', 'bathrooms', 'floor_size_clean', 'type_encoded', 'loc_Kiambu Road', 'loc_Kileleshwa', 'loc_Kilimani', 'loc_Lower Kabete', 'loc_Riverside', 'loc_Westlands']

Property type encoding:
  0 → Apartment
  1 → House
  2 → Townhouse


In [7]:
# Select final features for training
target_col = 'price_clean'

train_df = df[feature_cols + [target_col]].dropna()
print(f"Training dataset shape: {train_df.shape}")
print(train_df.head())

# Save
os.makedirs('../data', exist_ok=True)
train_df.to_csv('../data/train_ready.csv', index=False)

# Save encoders — only type encoder now (location is One-Hot, no encoder needed)
os.makedirs('../models', exist_ok=True)
with open('../models/encoders.pkl', 'wb') as f:
    pickle.dump({'property_type': le_type}, f)

print("\n✅ train_ready.csv saved")
print("✅ encoders.pkl saved")
print(f"Features used: {feature_cols}")

Training dataset shape: (66, 11)
   bedrooms  bathrooms  floor_size_clean  type_encoded  loc_Kiambu Road  \
0         4        4.0               211             0            False   
1         4        5.5               514             0            False   
3         2        2.0               140             0            False   
4         2        2.0                88             0            False   
6         3        4.5               230             0            False   

   loc_Kileleshwa  loc_Kilimani  loc_Lower Kabete  loc_Riverside  \
0           False         False             False          False   
1           False         False             False          False   
3           False         False             False          False   
4           False          True             False          False   
6           False         False             False          False   

   loc_Westlands  price_clean  
0           True     27000000  
1           True    127100000  
3          